# 15.1 - ML Evaluation Deep Dive

Status: VERIFIED

## What Are We Solving?
A model that you cannot measure is a model you cannot improve. Choosing the wrong metric can make a bad model look good and a good model look bad. This unit covers precision, recall, F1, ROC-AUC, calibration, and when each matters.

## Mental Model
Metrics are lenses. Accuracy is a wide-angle lens. Precision, recall, and F1 are zoom lenses that let you focus on specific failure types.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, precision_recall_curve, roc_curve,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
print("All imports OK")

All imports OK


## Core Concept: Confusion Matrix and Classification Metrics

In [2]:
# Generate imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=20, weights=[0.9, 0.1], random_state=42)
print(f"Class distribution: {np.bincount(y)}")
print(f"Majority class ratio: {np.bincount(y).max() / np.bincount(y).min():.1f}x")

# Train model
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

Class distribution: [897 103]
Majority class ratio: 8.7x



Confusion Matrix:
[[174   5]
 [ 13   8]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       179
           1       0.62      0.38      0.47        21

    accuracy                           0.91       200
   macro avg       0.77      0.68      0.71       200
weighted avg       0.90      0.91      0.90       200



## Why Accuracy Is Misleading on Imbalanced Data

In [3]:
# Accuracy looks great but the model barely catches fraud
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print(f"Accuracy:  {acc:.3f}  (looks good!)")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}  (catches only {rec*100:.0f}% of fraud)")
print(f"F1:        {f1:.3f}")
print(f"\n>>> Accuracy is {acc:.1%} but recall is only {rec:.1%} — the model misses most fraud!")

Accuracy:  0.910  (looks good!)
Precision: 0.615
Recall:    0.381  (catches only 38% of fraud)
F1:        0.471

>>> Accuracy is 91.0% but recall is only 38.1% — the model misses most fraud!


## ROC Curve and AUC

In [4]:
# ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

# Precision-recall curve
precision_arr, recall_arr, thresholds_pr = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(recall_arr, precision_arr, 'r-', linewidth=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"ROC AUC: {roc_auc:.3f}")

ROC AUC: 0.893


C:\Users\PC\AppData\Local\Temp\ipykernel_15892\275578912.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Threshold Tuning

In [5]:
# Different thresholds change precision/recall trade-off
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
print("Threshold | Precision | Recall | F1")
print("-" * 50)
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    print(f"   {t:.1f}    |   {p:.3f}   |  {r:.3f} | {f:.3f}")

print("\n>>> Lower threshold = more recall (catch more fraud) but lower precision")

Threshold | Precision | Recall | F1
--------------------------------------------------
   0.3    |   0.684   |  0.619 | 0.650
   0.4    |   0.688   |  0.524 | 0.595
   0.5    |   0.615   |  0.381 | 0.471
   0.6    |   0.600   |  0.286 | 0.387
   0.7    |   0.714   |  0.238 | 0.357

>>> Lower threshold = more recall (catch more fraud) but lower precision


## Regression Metrics

In [6]:
# Regression metrics demonstration
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
y_true_reg = np.random.uniform(100, 500, 100)
y_pred_reg = y_true_reg + np.random.normal(0, 30, 100)  # add noise

mae = mean_absolute_error(y_true_reg, y_pred_reg)
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)

print(f"MAE:  {mae:.2f}  (average absolute error)")
print(f"MSE:  {mse:.2f}  (penalizes large errors)")
print(f"RMSE: {rmse:.2f}  (same units as target)")
print(f"R2:   {r2:.3f}  (variance explained)")

MAE:  21.63  (average absolute error)
MSE:  742.60  (penalizes large errors)
RMSE: 27.25  (same units as target)
R2:   0.947  (variance explained)


## Error Analysis by Segment

In [7]:
# Find which samples the model fails on most
y_prob_test = model.predict_proba(X_test)[:, 1]
errors = np.abs(y_test - y_prob_test)
top_error_idx = np.argsort(errors)[-10:]

print("Top 10 most uncertain samples:")
print("Index | True | Prob   | Error")
print("-" * 40)
for idx in top_error_idx:
    print(f"  {idx:3d} |  {y_test[idx]}   | {y_prob_test[idx]:.3f} | {errors[idx]:.3f}")

# Feature importance for errors
print(f"\nMean feature values for errors vs correct:")
error_mask = errors > 0.5
print(f"  Error samples: {error_mask.sum()}")
print(f"  Correct samples: {(~error_mask).sum()}")

Top 10 most uncertain samples:
Index | True | Prob   | Error
----------------------------------------
   84 |  0   | 0.725 | 0.725
   14 |  1   | 0.208 | 0.792
   33 |  0   | 0.810 | 0.810
  166 |  1   | 0.140 | 0.860
   88 |  1   | 0.085 | 0.915
   66 |  1   | 0.074 | 0.926
  198 |  1   | 0.068 | 0.932
  165 |  1   | 0.059 | 0.941
   63 |  1   | 0.027 | 0.973
   41 |  1   | 0.006 | 0.994

Mean feature values for errors vs correct:
  Error samples: 18
  Correct samples: 182


In [8]:
# Verification
y_pred_final = model.predict(X_test)
assert accuracy_score(y_test, y_pred_final) > 0.8, "Accuracy too low"
assert f1_score(y_test, y_pred_final) > 0.3, "F1 too low"
print("VERIFICATION PASSED: Phase 15.1 complete")

VERIFICATION PASSED: Phase 15.1 complete
